# Ejercicio 3.

Se lanzan simultáneamente un par de dados legales y se anota el resultado de la suma de ambos. El proceso se repite hasta que todos los resultados posibles: $2, 3, \dots, 12$ hayan aparecido al menos una vez.

Estudiar mediante una simulación la variable $N$, el número de lanzamientos necesarios para cumplir el proceso. Cada lanzamiento implica arrojar el par de dados.

**a)** Describa la estructura lógica del
algoritmo que permite simular en computadora el número de lanzamientos necesarios para cumplir el proceso.

**b)** Mediante una implementación en computadora,

**(i)** estime el valor medio y la desviación estándar del número de lanzamientos, repitiendo el algoritmo: 100, 1000, 10000 y 100000 veces.

**(ii)** estime la probabilidad de que  $N$ sea por lo menos 15 y la probabilidad de que $N$ sea a lo sumo 9, repitiendo el algoritmo: 100, 1000, 10000 y 100000.

**a)** Estructura lógica del algoritmo

Queremos simular la variable aleatoria $N$: cantidad de lanzamientos necesarios hasta haber observado todas las sumas posibles de dos dados:

$$
\{2,3,4,\dots,12\}
$$

*Idea del algoritmo*

1. Inicializar el conjunto de sumas posibles.
2. Inicializar contador de lanzamientos: $N = 0$
3. *Mientras* el conjunto de sumas posibles sea distinto de vacío:
   - Generar dos números aleatorios uniformes en ${1, 2, 3, 4, 5, 6}$
   - Calcular la suma.
   - Si la suma pertence al conjunto de sumas posibles: eliminarla de dicho conjunto.
   - Incrementar el contador: $N = N + 1$
4. Devolver $N$.


In [ ]:
from random import random
import numpy as np
import statistics as stats

# Uniforme discreta en {1,...,n}
def udiscreta(n):
  U = random()
  return int(n * U) + 1

# Simulación del experimento
def simular_experimento():
  contador = 0
  resultados_posibles = list(range(2, 13))

  while len(resultados_posibles) != 0:  # len>0 => algunos valores no aparecieron
    dado1 = udiscreta(6)  # genero los dos dados
    dado2 = udiscreta(6)
    resultado = dado1 + dado2
    if resultado in resultados_posibles:  # si el res no fue generado antes
      resultados_posibles.remove(resultado)   # lo quito de la lista de posibles
    contador += 1
  return contador

# Repetir simulaciones para calcular estadisticas y probabilidades
def estadisticas(Nsim):
  resultados = []
  for _ in range(Nsim):
    resultados.append(simular_experimento())
  return stats.mean(resultados), stats.stdev(resultados)

def probabilidades(Nsim):
  prob1 = 0 # contador para resultados mayores o igaules a 15
  prob2 = 0 # contador para resultados menores o iguales a 9
  for n in range(Nsim):
    res = simular_experimento()
    if res >= 15:   # si el res es mayor o igual a 15 incremento el contador
      prob1 += 1
    if res <= 9:    # si el res es menor o igual a 9 incremento el contador
      prob2 += 1
  prob1 = prob1/Nsim  # para estimar probabilidades: favorables / posibles
  prob2 = prob2/Nsim
  return prob1, prob2

In [ ]:
# estimaciones:

Nsim = [100, 1000, 10000, 100000]

print(f"{'nsim':<10}{'media':<12}{'desvio':<12}{'P(N>=15)':<12}{'P(N<=9)':<12}")
print("-"*58)

for N in Nsim:
    media, desvio = estadisticas(N)
    p15, p9 = probabilidades(N)

    print(f"{N:<10}{media:<12.4f}{desvio:<12.4f}{p15:<12.4f}{p9:<12.4f}")

nsim      media       desvio      P(N>=15)    P(N<=9)     
----------------------------------------------------------
100       60.1300     34.4470     1.0000      0.0000      
1000      62.4130     37.7362     0.9960      0.0000      
10000     61.3688     36.5595     0.9990      0.0000      
100000    61.1861     35.8617     0.9987      0.0000      


# Ejercicio 4

Implemente cuatro métodos para generar una variable $X$ que toma los valores del $1$ al $10$, con probabilidades
 $p_1=0.11, \ p_2=0.14,\ p_3= 0.09,\  p_4=0.08,\  p_5=0.12,\ p_6= 0.10,\  p_7=0.09,\  p_8=0.07,\  p_9=0.11,\  p_{10}=0.09$ usando:


**a)** Método de rechazo con una uniforme discreta, buscando la cota $c$ más baja posible.

**b)** Método de rechazo con una uniforme discreta, usando $c = 3$.

**c)** Transformada inversa.

**d)** Método de la urna: utilizar un  arreglo $A$ de tamaño $100$ donde cada valor $i$ está en exactamente $p_i*100$ posiciones. El método debe devolver $A[k]$ con probabilidad $0.01$.
 ¿Por qué funciona?


 Compare la eficiencia de los tres algoritmos realizando $10000$ simulaciones.

La variable $X$ toma valores $1, 2, ..., 10$.

Si $p_j=P(X=j)$ con $j\in \{1,2,...,10 \}$ tenemos los siguientes datos:
$$
\begin{aligned}
p_1 &= 0.11, \quad p_2 = 0.14, \quad p_3 = 0.09, \quad p_4 = 0.08, \\
p_5 &= 0.12, \quad p_6 = 0.10, \quad p_7 = 0.09, \quad p_8 = 0.07, \\
p_9 &= 0.11, \quad p_{10} = 0.09
\end{aligned}
$$

In [ ]:
import numpy as np
import time

**Método de rechazo con una uniforme discreta, con $c$ lo más baja posible:**

Sea $Y \sim U(1,10)$, con $P(Y=j) = \frac{1}{10}$ para $j \in \{1, 2, \dots, 10\}$.

Quiero encontrar $c$ tal que:
$$ \frac{P(X=j)}{P(Y=j)} \leq c $$

Lo anterior vale si y sólo si:
$$ P(X=j) \leq c \cdot P(Y=j) = c \cdot \frac{1}{10}, \quad \forall j \in \{1, 2, \dots, 10\} $$

Luego, busco que $P(X=j) \leq \frac{c}{10}$.

Notemos que el valor con mayor probabilidad que puede tomar la v.a. $X$ es $2$ con $P(X=2) = 0.14$. Por lo tanto:
$$ 0.14 \leq \frac{c}{10} \implies c = 1.4 $$
donde $1.4$ es la mínima cota que cumple la desigualdad.

In [ ]:
# Si p(j) = P(X=j) y q(j) = P(Y=j)

# Simular Y
# U = random()
# if U < p(Y) / (c* q(Y)):
#   return Y                           --> aceptación
# else:
#   volver a paso 1                    --> rechazo

def udiscreta(n):
  u = np.random.random()
  return int(n*u)+1

def generoX_4a():
  p = [0.11, 0.14, 0.09, 0.08, 0.12, 0.10, 0.09, 0.07, 0.11, 0.09]
  while 1:
    y = udiscreta(10)
    u = np.random.random()
    cota = p[y-1] / 0.14     # c*q(y) = 1.4 * 0.1 = 0.14
    if u < cota:
      return y

def simulacion_4a(Nsim):
    inicio = time.time()
    for n in range(Nsim):
        _ = generoX_4a()
    fin = time.time()
    tiempo = fin - inicio
    return tiempo

In [ ]:
generoX_4a()

6

**Método de rechazo con una uniforme discreta, con $c=3$:**

In [ ]:
def udiscreta(n):
  u = np.random.random()
  return int(n*u)+1

def generoX_4b():
  p = [0.11, 0.14, 0.09, 0.08, 0.12, 0.10, 0.09, 0.07, 0.11, 0.09]
  while 1:
    y = udiscreta(10)
    u = np.random.random()
    cota = p[y-1] / 0.3     # c*q(y) = 3 * 0.1 = 0.3
    if u < cota:
      return y

def simulacion_4b(Nsim):
    inicio = time.time()
    for n in range(Nsim):
        _ = generoX_4b()
    fin = time.time()
    tiempo = fin - inicio
    return tiempo

In [ ]:
generoX_4b()

1

**Transformada inversa:**

In [ ]:
probs = [0.11, 0.14, 0.09, 0.08, 0.12, 0.10, 0.09, 0.07, 0.11, 0.09]
probs_ordenadas = sorted(probs, reverse=True)

print(probs_ordenadas)

[0.14, 0.12, 0.11, 0.11, 0.1, 0.09, 0.09, 0.09, 0.08, 0.07]


In [ ]:
def generoX_4c():
  u = np.random.random()
  if u < 0.14:
    return 2
  elif u < 0.26:
    return 5
  elif u < 0.37:
    return 1
  elif u < 0.48:
    return 9
  elif u < 0.58:
    return 6
  elif u < 0.67:
    return 3
  elif u < 0.76:
    return 7
  elif u < 0.85:
    return 10
  elif u < 0.93:
    return 4
  else:
    return 8

def simulacion_4c(Nsim):
    inicio = time.time()
    for n in range(Nsim):
        _ = generoX_4c()
    fin = time.time()
    tiempo = fin - inicio
    return tiempo

In [ ]:
generoX_4c()

2

In [ ]:
def generoX_4c_bis(probabilidades, valores):
    u = np.random.random()
    F = 0
    for i in range(len(probabilidades)):
        F += probabilidades[i]
        if u < F:
            return valores[i]

In [ ]:
probabilidades = [0.14, 0.12, 0.11, 0.11, 0.1, 0.09, 0.09, 0.09, 0.08, 0.07]
valores = [2, 5, 1, 9, 6, 10, 7, 3, 4, 8]

generoX_4c_bis(probabilidades, valores)

4

**Método de la urna:**

Quiero $k\in\mathbb{N}$ tal que $kp_j$ sea entero para cada $j\in \{1,2,...,10\}$.

Y considero un arreglo $A$ de $k$ posiciones donde se almacena cada valor $j$ en $kp_j$ posiciones  ($\sum_j kp_j = k \sum_j p_j = k $).



In [ ]:
def udiscreta(n):
  u = np.random.random()
  return int(n*u)             # sin el +1 genera números enteros en 0,1,2...,n-1.

def generoX_4d(A, n):
  I = udiscreta(n)
  return A[I]

def simulacion_4d(A, k, Nsim):
    inicio = time.time()
    for n in range(Nsim):
        _ = generoX_4d(A, k)
    fin = time.time()
    tiempo = fin - inicio
    return tiempo

In [ ]:
cantidades = [11, 14, 9, 8, 12, 10, 9, 7, 11, 9]   # cantidades = p * 100
A = []
for n in range(1,11):
  A += [n]*cantidades[n-1]

generoX_4d(A, len(A))

1

**Comparación de métodos:**

In [ ]:
Nsim = 10000
print('Tiempo método a: ', simulacion_4a(Nsim))
print('Tiempo método b: ', simulacion_4b(Nsim))
print('Tiempo método c: ', simulacion_4c(Nsim))
print('Tiempo método d: ', simulacion_4d(A, len(A), Nsim))

Tiempo método a:  0.028591632843017578
Tiempo método b:  0.04802656173706055
Tiempo método c:  0.00812840461730957
Tiempo método d:  0.008890867233276367
